## 0. Improvement Strategy
- From `exp_001`, it can be inferred that:
  - Model is overfitting.
  - The macro-averaged F1-score is still low.
  - A global threshold is inadequate; per-class thresholding is the correct approach.
  - Unfreezing all layers likely caused the model to overfit.

- Improvement decisions:
  - Loss function:
    - Before: `pos_weight` + `BCE`.
    - After: `pos_weight` + `BCE` (unchanged).
    - Rationale: focal loss combined with a raw `pos_weight` ranging up to 26.6x was considered and rejected. `pos_weight` scales every positive sample's loss regardless of difficulty, while the focal term only down-weights *easy* samples. For rare-class samples that are still hard to classify, the focal term does not soften anything and the large `pos_weight` is still applied in full -> the two mechanisms stack instead of complementing each other, risking unstable, over-penalized gradients on exactly the samples we most want to learn from. We keep the exp_001 baseline (`BCE+pos_weight`) and address overfitting through head/pooling and fine-tuning strategy instead.
  - Classifier head:
    - Before: single linear layer on top of mean-pooled patch embeddings (`hidden_dim=None`).
    - After: single linear layer on top of attention-pooled patch embeddings (learnable query computing softmax weights over patch tokens); `hidden_dim` kept at `None` (no added depth).
    - Rationale: Mean pooling averages all spatial regions with equal weight, which dilutes localized findings (fracture, small pneumothorax) against the much larger area of normal tissue in the same image. Attention pooling allows the model to learn which patch regions are most informative for the pathology classification task, effectively acting as a learnable spatial attention mechanism that highlights clinically relevant areas while suppressing irrelevant background.
  - Fine-tuning strategy:
    - Before: full backbone unfrozen and fine-tuned from epoch 1, single learning rate (1e-5) for both backbone and head.
    - After: gradual/progressive unfreezing with discriminative learning rates — backbone trained at 10x lower LR than the head.
    - Unfreezing schedule (epoch-based, verified against BioViL-T's actual module hierarchy via `named_children()`):
      - Epoch 1-2: Backbone FROZEN entirely (only head + attention pooling train). Head needs to stabilize first before backbone gets updated.
      - Epoch 3-5: `vit_pooler` (all 3 ViT blocks) + `backbone_to_vit` UNFROZEN. Smallest and most task-specific part (~1.2M params). Safe to adapt first.
      - Epoch 6-9: `layer4` UNFROZEN (~15M params). Held back until now because it's large — opening it too early would probably cause the same overfitting problem.
      - Epoch 10+: remaining layers UNFROZEN (`layer3`, `layer2`, `layer1`, stem). Full backbone active with very low LR.
      - Note: `projector` is excluded — not part of Stage-B's forward path. Confirmed by dimension mismatch: `projector`'s first layer expects 512-channel input while producing a 128-dim output, which doesn't line up with the classifier's actual `patch_embeddings` (512-dim, pre-projector); `projector` is used for BioViL-T's temporal/contrastive representation elsewhere in the pipeline, not here. Unfreezing it would do nothing since it never receives gradients through this forward path.
    - Rationale: freezing backbone for 2 epochs prevents head noise from ruining pretrained features. Unfreezing order is driven by parameter-count risk (small modules first, large ones later), not just depth. Discriminative LR (backbone 10x lower than head) keeps backbone updates gentle. Early stopping naturally caps adaptation depth — if performance plateaus early, training stops before larger layers are touched.
  - Usage of mixed precision (AMP) + gradient accumulation:
    - Before: full-precision (fp32) training, batch_size=16, single-step optimizer update per batch (~33 min/epoch).
    - After: AMP (`autocast` + `GradScaler`) enabled, gradient accumulation over 2 steps (effective batch size = 32).
    - Rationale: faster training iterations frees up time for more experimentation, and a larger effective batch size stabilizes gradients for BCE under heavy pos_weight, which otherwise produces noisy per-batch gradients at batch_size=16.
  - Train loop:
    - before: single metric monitoring on macro-averaged f1 score for earlystopping and checkpoint.
    - after: dual metric monitoring on macro-averaged f1 score and PR-AUC for earlystopping and checkpoint. each metric has its own patience counter; training stops when both metrics fail to improve for their respective patience. the model is saved independently for the best PR-AUC and the best macro-averaged f1 score.
    - rationale: PR-AUC evaluates ranking quality directly, independent of any cutoff choice (threshold-independent), avoiding bias of global default threshold during train loop.
  - Additional evaluation metrics:
    - Additional: PR-AUC on test set.
    - Additional: logging per-class f1 score on every epoch of train loop, plotted as a heatmap across epochs.
    - Additional: bootstrap confidence interval on test set evaluation metrics.
    - Rationale: the official test split is small (n=234), so point-estimate metrics are high-variance; a bootstrap CI communicates that uncertainty instead of implying a single number is precise.
    - best model on macro-averaged f1 score and best model on PR-AUC are evaluated separately.
  - Threshold tuning strategy:
    - Before: global threshold tuning.
    - After: per-pathology threshold tuning, grid-searched independently for each of the 13 classes on `dev_internal` with the goal of maximizing macro-averaged f1 score.
    - Rationale: the 13 sigmoid outputs come from independent neurons with uncalibrated magnitudes. Per-class thresholds allow each pathology to operate at its own optimal precision-recall point.
    - Perform on two models (best model on macro-averaged f1 score and best model on PR-AUC).
    - Rationale: since checkpoint selection is now dual-metric (F1 vs PR-AUC may not converge on the same epoch/weights), tuning thresholds on only one model would leave the other model's downstream performance uncharacterized. Running per-class threshold tuning on both keeps the comparison between the two checkpoints apples-to-apples all the way through to the final operating point, not just at the raw-probability stage.
  - New prompt injection strategy (implication of the new threshold tuning strategy):
    - Before: collect all pathologies with probability >= global threshold.
    - After: collect all pathologies with probability >= its own per-pathology threshold.
    - Rationale: preserves per-class calibration; prevents rare pathologies from being suppressed by a globally conservative threshold. A pathology is included in the prompt if and only if its predicted probability >= its individually-tuned threshold.
    - Open decision (not yet resolved by this document): with per-class thresholds now tuned for both the best-F1 and best-PR-AUC checkpoints, it is not yet specified which model's thresholds actually drive the production `"chexpert findings: {list}"` string. Carrying the earlier convention forward, the default assumption implemented in this notebook is that the best-PR-AUC checkpoint + its per-class thresholds is the one substituted into prompt injection, with the best-F1 checkpoint's thresholds kept only as a comparison artifact — this should be confirmed explicitly rather than inherited silently, since it directly determines what Stage-MEETING receives as input.

## 1. Import Library and Setup Configuration

In [ ]:
# import necessary libraries
import os
import json
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, balanced_accuracy_score, f1_score, average_precision_score
from health_multimodal.image import get_image_inference
from health_multimodal.image.utils import ImageModelType

In [ ]:
# setup reproducibility
def seed_everything(seed=42):
    import random
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

In [ ]:
# setup configuration
config = {
    "experiment": "exp_001",
    "stage": "stage_b",

    "data": {
        "train_path": r"D:\VLM-Research_Task-C\data\processed\chexpert_plus\train.parquet",
        "dev_path": r"D:\VLM-Research_Task-C\data\processed\chexpert_plus\dev_internal.parquet",
        "test_path": r"D:\VLM-Research_Task-C\data\processed\chexpert_plus\test.parquet",
        "image_col": "actual_image_path",
        "label_suffix": "_label"
    },

    "model": {
        "freeze_backbone": False,          # control freezing via unfreeze_schedule
        "hidden_dim": None,                # single linear layer (no added depth)
        "num_pathologies": 14,
        "dropout": 0.1,
        "pooling": "attention"             # attention pooling over patch tokens
    },

    "train": {
        "batch_size": 16,
        "num_epochs": 30,
        "lr": 1.0e-5,                      # head learning rate
        "backbone_lr_mult": 0.1,           # backbone lr = lr * backbone_lr_mult
        "weight_decay": 1.0e-4,
        "grad_clip_norm": 1.0,
        "early_stopping_patience": 5,      # patience for both f1 and pr-auc
        "num_workers": 0,
        "seed": 42,
        "lr_scheduler": "ReduceLROnPlateau",
        "lr_scheduler_patience": 3,
        "use_amp": True,                   # mixed precision training
        "grad_accum_steps": 2,             # effective batch size = 16 * 2 = 32
        "unfreeze_schedule": {
            1: [],                         # epoch 1-2: backbone frozen
            3: ["vit_pooler", "backbone_to_vit"],   # epoch 3-5: small modules
            6: ["vit_pooler", "backbone_to_vit", "layer4"],  # epoch 6-9: add layer4
            10: ["vit_pooler", "backbone_to_vit", "layer4", "layer3", "layer2", "layer1", "stem"]  # epoch 10+: full
        }
    },

    "loss": {
        "type": "bce" # binary cross-entropy with pos_weight
    },

    "eval": {
        "bootstrap_n": 1000,
        "bootstrap_ci": 0.95
    },

    "output": {
        "checkpoint_dir": r"D:\VLM-Research_Task-C\output\exp_001\stage_b\checkpoints",
        "log_dir": r"D:\VLM-Research_Task-C\output\exp_001\stage_b\logs",
        "tuning_dir": r"D:\VLM-Research_Task-C\output\exp_001\stage_b\tuning",
        "plot_dir": r"D:\VLM-Research_Task-C\output\exp_001\stage_b\plots",
        "best_f1_filename": "stage_b_best_f1.pt",
        "best_prauc_filename": "stage_b_best_prauc.pt"
    },

    "device": "cuda"
}

# create output directories
for key in ["checkpoint_dir", "log_dir", "tuning_dir", "plot_dir"]:
    os.makedirs(config["output"][key], exist_ok=True)

In [ ]:
# define device, if GPU doesn't exist, it'll fallback to CPU usage
device = torch.device(config['device'] if torch.cuda.is_available() else 'cpu')
print(f"using device: {device}")

## 2. Load Data

In [ ]:
# load into a dataframe for each set
df_train = pd.read_parquet(config['data']['train_path'])
df_dev = pd.read_parquet(config['data']['dev_path'])
df_test = pd.read_parquet(config['data']['test_path'])

In [ ]:
# quick exploration

# dataset dimension
print(f"train set: {df_train.shape[0]} rows & {df_train.shape[1]} columns")
print(f"dev set: {df_dev.shape[0]} rows & {df_dev.shape[1]} columns")
print(f"test set: {df_test.shape[0]} rows & {df_test.shape[1]} columns")

# extract CXR pathology label columns
label_cols = [col for col in df_train.columns if col.endswith(config['data']['label_suffix'])]
print(f"label columns ({len(label_cols)}): {label_cols}")

# make sure there are 14 CXR pathology labels
assert len(label_cols) == 14, "there have to be 14 CXR pathology labels"

## 3. Training Utilities

### 3.1 Dataset Class

In [ ]:
class CheXpertPlusDataset(Dataset):
    def __init__(self, df, image_col, label_cols, transform=None):
        self.df = df.reset_index(drop=True)
        self.image_col = image_col
        self.label_cols = label_cols
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row[self.image_col]
        try:
            image = Image.open(img_path).convert('L')
        except FileNotFoundError:
            print(f"warning: image not found {img_path}, returning zeros.")
            image = Image.new('L', (224, 224))
        if self.transform:
            image = self.transform(image)
        labels = torch.tensor(row[self.label_cols].values.astype(np.float32), dtype=torch.float32)
        return image, labels

### 3.2 Load BioViL-T Image Encoder

In [ ]:
# load biovil-t image encoder
image_engine = get_image_inference(ImageModelType.BIOVIL_T)
biovilt_transform = image_engine.transform
bio_model = image_engine.model
bio_model.to(device)
bio_model.eval()

print("biovil-t loaded successfully")

In [ ]:
# inspect the original layer structure of BioViL-T (for unfreeze schedule)
print("bio_model top-level children:")
for name, _ in bio_model.named_children():
    print(f"  {name}")

print("\nbio_model.encoder children:")
for name, _ in bio_model.encoder.named_children():
    print(f"  {name}")

print("\nbio_model.encoder.encoder (resnet) children:")
for name, _ in bio_model.encoder.encoder.named_children():
    print(f"  {name}")

In [ ]:
# detect vision hidden dimension
sample_row = df_train.iloc[0]
sample_img_path = sample_row[config['data']['image_col']]
try:
    sample_img = Image.open(sample_img_path).convert('L')
except FileNotFoundError:
    sample_img = Image.new('L', (224, 224))

sample_pixel_values = biovilt_transform(sample_img).unsqueeze(0).to(device)

with torch.no_grad():
    dummy_out = bio_model(sample_pixel_values)

    # extract features tensor from dummy_out
    if hasattr(dummy_out, 'patch_embeddings'):
        features = dummy_out.patch_embeddings
        print("using patch_embeddings")
    elif hasattr(dummy_out, 'last_hidden_state'):
        features = dummy_out.last_hidden_state
        print("using last_hidden_state")
    else:
        # if dummy_out is a tensor directly, use it
        if isinstance(dummy_out, torch.Tensor):
            features = dummy_out
            print("using raw tensor output")
        else:
            # fallback: try to get encoder output
            try:
                features = bio_model.encoder(sample_pixel_values)
                print("using encoder output")
            except:
                raise RuntimeError("Cannot extract features from BioViL-T output")

    # determine vision dimension based on features shape
    if features.dim() == 3:
        # (B, seq, D)
        vision_dim = features.shape[-1]
    elif features.dim() == 4:
        # (B, C, H, W)
        vision_dim = features.shape[1]
    else:
        # fallback
        vision_dim = features.shape[-1]

    # sanity check: if vision_dim is suspiciously small (like 14), set to a known good value
    if vision_dim < 100:
        print(f"warning: detected vision_dim={vision_dim}, which is suspiciously small. forcing to 768 (default BioViL-T dimension).")
        vision_dim = 768

config['vision_dim'] = vision_dim
print(f"vision hidden dimension: {vision_dim}")

### 3.3 Create Dataset & Dataloader

In [ ]:
train_dataset = CheXpertPlusDataset(df_train, config['data']['image_col'], label_cols, transform=biovilt_transform)
dev_dataset = CheXpertPlusDataset(df_dev, config['data']['image_col'], label_cols, transform=biovilt_transform)
test_dataset = CheXpertPlusDataset(df_test, config['data']['image_col'], label_cols, transform=biovilt_transform)

train_loader = DataLoader(train_dataset, batch_size=config['train']['batch_size'], shuffle=True, num_workers=config['train']['num_workers'], pin_memory=True)
dev_loader = DataLoader(dev_dataset, batch_size=config['train']['batch_size'], shuffle=False, num_workers=config['train']['num_workers'], pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=config['train']['batch_size'], shuffle=False, num_workers=config['train']['num_workers'], pin_memory=True)

print(f"train batches: {len(train_loader)}, dev batches: {len(dev_loader)}, test batches: {len(test_loader)}")

### 3.4 Compute Weights to Mitigate Imbalanced Class

In [ ]:
def compute_pos_weights(df: pd.DataFrame, label_cols: list) -> torch.Tensor:
    """
    per-pathology pos_weight for nn.BCEWithLogitsLoss, from train prevalence only.
    pos_weight_i = num_negative_i / num_positive_i.
    """
    pos_counts = (df[label_cols] == 1.0).sum()
    neg_counts = (df[label_cols] == 0.0).sum()
    pos_weight = (neg_counts / pos_counts.clip(lower=1)).to_numpy(dtype="float32")
    return torch.tensor(pos_weight)

pos_weight = compute_pos_weights(df_train, label_cols).to(device)

print("class weights (pos_weight = N_neg / N_pos):")
for col, w in zip(label_cols, pos_weight.cpu().numpy()):
    print(f"  {col}: {w:.3f}")

In [ ]:
# loss function (BCE with pos_weight)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

### 3.5 Define Stage-B Classifier Model

In [ ]:
class StageBClassifier(nn.Module):
    """
    classifier using BioViL-T backbone + optional attention pooling over patch tokens.
    the attention pooling replaces mean pooling to focus on informative spatial regions.
    """
    def __init__(self, config, backbone, vision_dim, freeze_backbone=False):
        super().__init__()
        self.backbone = backbone
        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False

        self.hidden_dim = config['model']['hidden_dim']
        self.num_classes = config['model']['num_pathologies']
        self.dropout_rate = config['model']['dropout']
        self.pooling = config['model']['pooling']

        in_features = vision_dim

        # attention pooling layer: learns a weight per patch
        if self.pooling == "attention":
            self.attn_pool = nn.Linear(in_features, 1)

        # classifier head (single linear layer if hidden_dim is None)
        if self.hidden_dim is not None:
            self.classifier = nn.Sequential(
                nn.Dropout(self.dropout_rate),
                nn.Linear(in_features, self.hidden_dim),
                nn.ReLU(),
                nn.Dropout(self.dropout_rate),
                nn.Linear(self.hidden_dim, self.num_classes)
            )
        else:
            self.classifier = nn.Sequential(
                nn.Dropout(self.dropout_rate),
                nn.Linear(in_features, self.num_classes)
            )

    def forward(self, x):
        out = self.backbone(x)

        # extract features
        if hasattr(out, 'patch_embeddings'):
            features = out.patch_embeddings
        elif hasattr(out, 'last_hidden_state'):
            features = out.last_hidden_state
        else:
            features = out

        if not isinstance(features, torch.Tensor):
            raise TypeError(f"Expected Tensor, got {type(features)}")

        # pooling: reduce to (B, D)
        if features.dim() == 3:
            if self.pooling == "attention":
                attn_scores = self.attn_pool(features)              # (B, seq, 1)
                attn_weights = torch.softmax(attn_scores, dim=1)    # (B, seq, 1)
                pooled = (features * attn_weights).sum(dim=1)       # (B, D)
            else:
                pooled = features.mean(dim=1)  # mean pooling
        elif features.dim() == 4:
            if self.pooling == "attention":
                b, c, h, w = features.shape
                flat = features.flatten(2).transpose(1, 2)          # (B, H*W, C)
                attn_scores = self.attn_pool(flat)                  # (B, H*W, 1)
                attn_weights = torch.softmax(attn_scores, dim=1)
                pooled = (flat * attn_weights).sum(dim=1)            # (B, C)
            else:
                pooled = features.mean(dim=[2, 3])  # global average pooling
        else:
            pooled = features.flatten(1) if features.dim() > 2 else features

        logits = self.classifier(pooled)
        return logits

In [ ]:
# instantiate model
model = StageBClassifier(
    config,
    backbone=bio_model,
    vision_dim=config['vision_dim'],
    freeze_backbone=config['model']['freeze_backbone']  # will be overridden by schedule
)
model = model.to(device)

print(f"model initialized on {device}")
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"total parameters: {total_params/1e6:.2f}M, trainable: {trainable_params/1e6:.2f}M")

### 3.6 Gradual Unfreezing Utility

In [ ]:
# mapping: group name -> original module(s)
def get_layer_group_modules(model):
    resnet = model.backbone.encoder.encoder
    return {
        "stem": [resnet.conv1, resnet.bn1],
        "layer1": [resnet.layer1],
        "layer2": [resnet.layer2],
        "layer3": [resnet.layer3],
        "layer4": [resnet.layer4],
        "backbone_to_vit": [model.backbone.encoder.backbone_to_vit],
        "vit_pooler": [model.backbone.encoder.vit_pooler],
    }

def apply_unfreeze_schedule(model, unfrozen_group_names):
    """set requires_grad for the entire backbone based on the groups to unfreeze."""
    groups = get_layer_group_modules(model)
    for group_name, modules in groups.items():
        trainable = group_name in unfrozen_group_names
        for module in modules:
            for param in module.parameters():
                param.requires_grad = trainable

def get_unfrozen_groups_for_epoch(schedule, epoch):
    """find the latest active schedule whose epoch is <= the current epoch."""
    applicable_epochs = [e for e in schedule.keys() if e <= epoch]
    active_epoch = max(applicable_epochs)
    return schedule[active_epoch]

# validate the schedule against the actual model structure to fail early on typos
_valid_group_names = set(get_layer_group_modules(model).keys())
for _epoch, _groups in config['train']['unfreeze_schedule'].items():
    assert set(_groups).issubset(_valid_group_names), f"unknown group name in unfreeze_schedule at epoch {_epoch}: {_groups}"
print("unfreeze schedule validated against actual model structure")

## 4. Train Loop

In [ ]:
# find no finding index, exclude from metrics
no_finding_idx = None
for i, col in enumerate(label_cols):
    if col.lower().startswith('no finding'):
        no_finding_idx = i
        break
if no_finding_idx is None:
    no_finding_idx = 0
other_indices = [i for i in range(len(label_cols)) if i != no_finding_idx]

print(f"no finding index: {no_finding_idx}")
print(f"other indices (13 pathologies): {other_indices}")

In [ ]:
# ensure all backbone parameters are frozen before epoch 1
apply_unfreeze_schedule(model, get_unfrozen_groups_for_epoch(config['train']['unfreeze_schedule'], epoch=1))
print(f"epoch 1 backbone groups unfrozen: {get_unfrozen_groups_for_epoch(config['train']['unfreeze_schedule'], epoch=1)}")

In [ ]:
# differential learning rates: backbone trained more slowly than head
head_params = list(model.classifier.parameters())
if config['model']['pooling'] == "attention":
    head_params += list(model.attn_pool.parameters())
backbone_params = list(model.backbone.parameters())

optimizer = torch.optim.AdamW(
    [
        {"params": backbone_params, "lr": config['train']['lr'] * config['train']['backbone_lr_mult']},
        {"params": head_params, "lr": config['train']['lr']}
    ],
    weight_decay=config['train']['weight_decay']
)

# learning rate scheduler (reduces lr when loss plateaus)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=config['train']['lr_scheduler_patience'], factor=0.5
)

# automatic mixed precision scaler
scaler = torch.amp.GradScaler('cuda', enabled=config['train']['use_amp'])

# clear gpu cache before starting
torch.cuda.empty_cache()

In [ ]:
# training loop with dual metric monitoring (f1 and pr-auc)
best_val_macro_f1 = 0.0
best_val_macro_prauc = 0.0
early_stop_counter_f1 = 0
early_stop_counter_prauc = 0

train_losses, val_losses = [], []
train_macro_f1s, val_macro_f1s, val_macro_praucs = [], [], []
val_per_class_f1_history = []

current_unfrozen_groups = get_unfrozen_groups_for_epoch(config['train']['unfreeze_schedule'], epoch=1)

print("\nstarting training (optimizer: adamw, dual early stopping on macro-f1 & pr-auc, 13 pathologies excluding no finding)")
for epoch in range(1, config['train']['num_epochs'] + 1):

    # --- apply scheduled unfreezing at the start of each epoch ---
    scheduled_groups = get_unfrozen_groups_for_epoch(config['train']['unfreeze_schedule'], epoch)
    if scheduled_groups != current_unfrozen_groups:
        apply_unfreeze_schedule(model, scheduled_groups)
        current_unfrozen_groups = scheduled_groups
        print(f"epoch {epoch}: backbone groups unfrozen updated -> {current_unfrozen_groups}")

    # --- training phase ---
    model.train()
    train_loss = 0.0
    train_preds, train_labels = [], []
    optimizer.zero_grad()
    for step, (images, labels) in enumerate(tqdm(train_loader, desc=f"epoch {epoch}/{config['train']['num_epochs']} (train)")):
        images, labels = images.to(device), labels.to(device)

        with torch.amp.autocast('cuda', enabled=config['train']['use_amp']):
            logits = model(images)
            loss = criterion(logits, labels) / config['train']['grad_accum_steps']

        scaler.scale(loss).backward()

        # gradient accumulation: update every grad_accum_steps
        if (step + 1) % config['train']['grad_accum_steps'] == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), config['train']['grad_clip_norm'])
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        train_loss += loss.item() * config['train']['grad_accum_steps'] * images.size(0)

        probs = torch.sigmoid(logits)
        train_preds.append(probs.detach().cpu().numpy())
        train_labels.append(labels.detach().cpu().numpy())
    train_loss /= len(train_loader.dataset)
    train_losses.append(train_loss)

    # compute train macro-f1 on 13 pathologies (excluding no finding)
    train_preds = np.vstack(train_preds)
    train_labels = np.vstack(train_labels)
    train_preds_binary = (train_preds > 0.5).astype(int)
    train_preds_other = train_preds_binary[:, other_indices]
    train_labels_other = train_labels[:, other_indices]
    train_macro_f1 = f1_score(train_labels_other, train_preds_other, average='macro', zero_division=0)
    train_macro_f1s.append(train_macro_f1)

    # --- validation phase ---
    model.eval()
    val_loss = 0.0
    val_preds, val_labels = [], []
    with torch.no_grad():
        for images, labels in tqdm(dev_loader, desc=f"epoch {epoch}/{config['train']['num_epochs']} (val)"):
            images, labels = images.to(device), labels.to(device)
            with autocast(enabled=config['train']['use_amp']):
                logits = model(images)
                loss = criterion(logits, labels)
            val_loss += loss.item() * images.size(0)
            probs = torch.sigmoid(logits)
            val_preds.append(probs.cpu().numpy())
            val_labels.append(labels.cpu().numpy())
    val_loss /= len(dev_loader.dataset)
    val_losses.append(val_loss)

    # compute val macro-f1 (threshold=0.5) on 13 pathologies
    val_preds = np.vstack(val_preds)
    val_labels = np.vstack(val_labels)
    val_preds_binary = (val_preds > 0.5).astype(int)
    val_preds_other = val_preds_binary[:, other_indices]
    val_labels_other = val_labels[:, other_indices]
    val_macro_f1 = f1_score(val_labels_other, val_preds_other, average='macro', zero_division=0)
    val_macro_f1s.append(val_macro_f1)

    # compute val macro pr-auc (threshold-independent) on 13 pathologies
    val_probs_other = val_preds[:, other_indices]
    per_class_ap = [
        average_precision_score(val_labels_other[:, i], val_probs_other[:, i])
        for i in range(val_labels_other.shape[1])
    ]
    val_macro_prauc = np.mean(per_class_ap)
    val_macro_praucs.append(val_macro_prauc)

    # log per-class f1 each epoch to detect rare-class collapse early
    per_class_f1 = f1_score(val_labels_other, val_preds_other, average=None, zero_division=0)
    val_per_class_f1_history.append(per_class_f1)
    worst_idx = np.argsort(per_class_f1)[:3]
    worst_report = ", ".join(f"{label_cols[other_indices[i]]}={per_class_f1[i]:.3f}" for i in worst_idx)

    print(f"epoch {epoch}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}, "
          f"train_macro_f1={train_macro_f1:.4f}, val_macro_f1={val_macro_f1:.4f}, val_macro_prauc={val_macro_prauc:.4f}")
    print(f"-> 3 weakest classes (val f1): {worst_report}")

    # scheduler step based on val loss
    scheduler.step(val_loss)

    # --- checkpoint and early stopping for f1 (independent) ---
    if val_macro_f1 > best_val_macro_f1:
        best_val_macro_f1 = val_macro_f1
        early_stop_counter_f1 = 0
        best_f1_path = os.path.join(config['output']['checkpoint_dir'], config['output']['best_f1_filename'])
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(), 'best_val_macro_f1': best_val_macro_f1}, best_f1_path)
        print(f"  -> new best-by-f1 model saved (val_macro_f1={val_macro_f1:.4f})")
    else:
        early_stop_counter_f1 += 1

    # --- checkpoint and early stopping for pr-auc (independent) ---
    if val_macro_prauc > best_val_macro_prauc:
        best_val_macro_prauc = val_macro_prauc
        early_stop_counter_prauc = 0
        best_prauc_path = os.path.join(config['output']['checkpoint_dir'], config['output']['best_prauc_filename'])
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(), 'best_val_macro_prauc': best_val_macro_prauc}, best_prauc_path)
        print(f"  -> new best-by-prauc model saved (val_macro_prauc={val_macro_prauc:.4f})")
    else:
        early_stop_counter_prauc += 1

    # stop only when both metrics fail to improve for their respective patience
    if early_stop_counter_f1 >= config['train']['early_stopping_patience'] and \
       early_stop_counter_prauc >= config['train']['early_stopping_patience']:
        print(f"early stopping triggered at epoch {epoch} (both f1 and pr-auc stalled for {config['train']['early_stopping_patience']} epochs)")
        break

print(f"\ntraining finished. best val macro-f1: {best_val_macro_f1:.4f} | best val macro pr-auc: {best_val_macro_prauc:.4f}")

## 5. Model Evaluation

In [ ]:
plt.figure(figsize=(16, 4))

plt.subplot(1, 3, 1)
plt.plot(train_losses, label='train loss')
plt.plot(val_losses, label='val loss')
plt.xlabel('epoch')
plt.ylabel('loss')
plt.legend()
plt.title('training & validation loss (bce)')
plt.grid(True)

plt.subplot(1, 3, 2)
plt.plot(train_macro_f1s, label='train macro-f1 (13 pathologies)')
plt.plot(val_macro_f1s, label='val macro-f1 (13 pathologies)')
plt.xlabel('epoch')
plt.ylabel('macro-f1 (threshold=0.5)')
plt.legend()
plt.title('training & validation macro-f1')
plt.grid(True)

plt.subplot(1, 3, 3)
plt.plot(val_macro_praucs, label='val macro pr-auc (13 pathologies)', color='green')
plt.xlabel('epoch')
plt.ylabel('macro pr-auc')
plt.legend()
plt.title('validation macro pr-auc (threshold-independent)')
plt.grid(True)

plt.tight_layout()
plt.savefig(os.path.join(config['output']['plot_dir'], 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# per-class f1 heatmap over epochs
per_class_f1_matrix = np.vstack(val_per_class_f1_history)
other_label_names = [label_cols[i] for i in other_indices]

plt.figure(figsize=(10, 6))
sns.heatmap(
    per_class_f1_matrix.T,
    xticklabels=range(1, len(val_per_class_f1_history) + 1),
    yticklabels=other_label_names,
    cmap='viridis',
    annot=False
)
plt.xlabel('epoch')
plt.title('val f1 per pathology sepanjang epoch (excluding no finding)')
plt.tight_layout()
plt.savefig(os.path.join(config['output']['plot_dir'], 'per_class_f1_history.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
def bootstrap_metric_ci(labels, preds_binary, metric_fn, n_boot=1000, ci=0.95, seed=42):
    """bootstrap confidence interval for a given metric (e.g., balanced accuracy)."""
    rng = np.random.default_rng(seed)
    n_samples = labels.shape[0]
    boot_scores = []
    for _ in range(n_boot):
        idx = rng.integers(0, n_samples, size=n_samples)
        boot_labels = labels[idx]
        boot_preds = preds_binary[idx]
        per_class = [
            metric_fn(boot_labels[:, i], boot_preds[:, i])
            for i in range(boot_labels.shape[1])
            if len(np.unique(boot_labels[:, i])) > 1  # skip classes that happen to be single-class in this resample
        ]
        if len(per_class) > 0:
            boot_scores.append(np.mean(per_class))
    lower = np.percentile(boot_scores, (1 - ci) / 2 * 100)
    upper = np.percentile(boot_scores, (1 + ci) / 2 * 100)
    return np.mean(boot_scores), lower, upper

def evaluate_checkpoint_on_test(checkpoint_path, label):
    """load a checkpoint and evaluate on test set, returning metrics and predictions."""
    model.load_state_dict(torch.load(checkpoint_path)['model_state_dict'])
    model.eval()

    test_preds, test_labels = [], []
    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc=f"evaluating test set ({label})"):
            images = images.to(device)
            logits = model(images)
            probs = torch.sigmoid(logits)
            test_preds.append(probs.cpu().numpy())
            test_labels.append(labels.cpu().numpy())

    test_preds = np.vstack(test_preds)
    test_labels = np.vstack(test_labels)
    test_preds_binary = (test_preds > 0.5).astype(int)

    print(f"\n=== test set performance ({label}, threshold=0.5) ===")
    print(classification_report(test_labels, test_preds_binary, target_names=label_cols, zero_division=0))

    bal_accs = [balanced_accuracy_score(test_labels[:, i], test_preds_binary[:, i]) for i in range(test_labels.shape[1])]
    test_bal_acc = np.mean(bal_accs)

    test_praucs = [average_precision_score(test_labels[:, i], test_preds[:, i]) for i in range(test_labels.shape[1])]
    test_macro_prauc = np.mean(test_praucs)

    boot_mean, boot_lo, boot_hi = bootstrap_metric_ci(
        test_labels, test_preds_binary, balanced_accuracy_score,
        n_boot=config['eval']['bootstrap_n'], ci=config['eval']['bootstrap_ci']
    )

    print(f"bal_acc: {test_bal_acc:.4f} | macro pr-auc: {test_macro_prauc:.4f}")
    print(f"bal_acc bootstrap {int(config['eval']['bootstrap_ci']*100)}% CI: [{boot_lo:.4f}, {boot_hi:.4f}] (mean={boot_mean:.4f}, n_boot={config['eval']['bootstrap_n']})")

    return {
        "test_preds": test_preds, "test_labels": test_labels, "test_preds_binary": test_preds_binary,
        "bal_acc": test_bal_acc, "macro_prauc": test_macro_prauc,
        "bal_acc_ci": (boot_lo, boot_hi), "bal_acc_boot_mean": boot_mean
    }

In [ ]:
# evaluate both checkpoints on test set
best_f1_path = os.path.join(config['output']['checkpoint_dir'], config['output']['best_f1_filename'])
best_prauc_path = os.path.join(config['output']['checkpoint_dir'], config['output']['best_prauc_filename'])

results_best_f1 = evaluate_checkpoint_on_test(best_f1_path, "best-by-f1")
results_best_prauc = evaluate_checkpoint_on_test(best_prauc_path, "best-by-prauc")

## 6. Threshold Tuning

In [ ]:
# threshold tuning (global vs per-pathology) on dev set for both checkpoints
thresholds = np.arange(0.1, 0.91, 0.05)

def extract_dev_probs(checkpoint_path, label):
    """extract probabilities and labels from dev set for a given checkpoint."""
    model.load_state_dict(torch.load(checkpoint_path)['model_state_dict'])
    model.eval()
    probs_list, labels_list = [], []
    with torch.no_grad():
        for images, labels in tqdm(dev_loader, desc=f"extracting dev probabilities ({label})"):
            images = images.to(device)
            logits = model(images)
            probs = torch.sigmoid(logits)
            probs_list.append(probs.cpu().numpy())
            labels_list.append(labels.cpu().numpy())
    return np.vstack(probs_list), np.vstack(labels_list)

def tune_thresholds(dev_probs, dev_labels, label_cols, other_indices, thresholds, label):
    """tune global threshold (baseline) and per-pathology thresholds (used for prompt injection)."""
    best_th, best_macro_f1 = 0.5, 0.0
    for th in thresholds:
        preds_th = (dev_probs[:, other_indices] >= th).astype(int)
        labels_th = dev_labels[:, other_indices]
        macro_f1 = f1_score(labels_th, preds_th, average='macro', zero_division=0)
        if macro_f1 > best_macro_f1:
            best_macro_f1 = macro_f1
            best_th = th

    # per-pathology threshold (used for prompt injection)
    per_class_th, per_class_f1 = {}, {}
    for global_i in other_indices:
        col_name = label_cols[global_i]
        col_probs = dev_probs[:, global_i]
        col_labels = dev_labels[:, global_i]
        best_col_th, best_col_f1 = 0.5, 0.0
        for th in thresholds:
            f1 = f1_score(col_labels, (col_probs >= th).astype(int), zero_division=0)
            if f1 > best_col_f1:
                best_col_f1 = f1
                best_col_th = th
        per_class_th[col_name] = float(best_col_th)
        per_class_f1[col_name] = float(best_col_f1)

    macro_f1_per_class_th = np.mean(list(per_class_f1.values()))
    print(f"[{label}] best global threshold: {best_th:.2f} (macro f1: {best_macro_f1:.4f})")
    print(f"[{label}] macro f1 with per-class thresholds: {macro_f1_per_class_th:.4f}")
    return {
        "best_th": best_th, "best_macro_f1": best_macro_f1,
        "per_class_th": per_class_th, "per_class_f1": per_class_f1,
        "macro_f1_per_class_th": macro_f1_per_class_th
    }

In [ ]:
# extract dev probabilities for both checkpoints
dev_probs_f1, dev_labels_f1 = extract_dev_probs(best_f1_path, "best-by-f1")
dev_probs_prauc, dev_labels_prauc = extract_dev_probs(best_prauc_path, "best-by-prauc")

print("\nthreshold tuning - best-by-f1 checkpoint")
tuning_f1 = tune_thresholds(dev_probs_f1, dev_labels_f1, label_cols, other_indices, thresholds, "best-by-f1")

print("\nthreshold tuning - best-by-prauc checkpoint")
tuning_prauc = tune_thresholds(dev_probs_prauc, dev_labels_prauc, label_cols, other_indices, thresholds, "best-by-prauc")

# save tuning results for each checkpoint
for label, tuning, dev_probs_x, dev_labels_x in [
    ("best_f1", tuning_f1, dev_probs_f1, dev_labels_f1),
    ("best_prauc", tuning_prauc, dev_probs_prauc, dev_labels_prauc),
]:
    with open(os.path.join(config['output']['tuning_dir'], f'best_threshold_global_{label}.txt'), 'w') as f:
        f.write(str(tuning["best_th"]))
    with open(os.path.join(config['output']['tuning_dir'], f'best_threshold_per_class_{label}.json'), 'w') as f:
        json.dump(tuning["per_class_th"], f, indent=2)
    np.save(os.path.join(config['output']['tuning_dir'], f'dev_probs_{label}.npy'), dev_probs_x)
    np.save(os.path.join(config['output']['tuning_dir'], f'dev_labels_{label}.npy'), dev_labels_x)

In [ ]:
# normal detection & abnormal recall validation on dev set (best-by-prauc + per-class)
def flag_pathologies(row_probs, per_class_th, label_cols, other_indices):
    """return indices of pathologies that exceed their per-class threshold."""
    return [i for i in other_indices if row_probs[i] >= per_class_th[label_cols[i]]]

# normal cases
truly_normal_mask = df_dev['No Finding_label'] == 1.0
truly_normal_probs = dev_probs_prauc[truly_normal_mask]
correctly_flagged_normal = sum(
    1 for row_probs in truly_normal_probs
    if len(flag_pathologies(row_probs, tuning_prauc["per_class_th"], label_cols, other_indices)) == 0
)
total_normal = len(truly_normal_probs)
normal_detection_rate = correctly_flagged_normal / total_normal if total_normal > 0 else 0.0
print("\nvalidation on genuinely normal cases (dev set, best-by-prauc checkpoint), per-class threshold:")
print(f"total genuinely normal samples in dev: {total_normal}")
print(f"correctly flagged as 'no significant findings': {correctly_flagged_normal}")
print(f"normal detection rate: {normal_detection_rate:.2%}")

# abnormal cases
abnormal_mask = (df_dev[label_cols].drop(columns=['No Finding_label']).sum(axis=1) > 0)
abnormal_probs = dev_probs_prauc[abnormal_mask]
abnormal_detected = sum(
    1 for row_probs in abnormal_probs
    if len(flag_pathologies(row_probs, tuning_prauc["per_class_th"], label_cols, other_indices)) > 0
)
total_abnormal = len(abnormal_probs)
abnormal_recall_rate = abnormal_detected / total_abnormal if total_abnormal > 0 else 0.0
covered = total_normal + total_abnormal
uncovered = len(df_dev) - covered
print(f"total genuinely abnormal samples in dev: {total_abnormal}")
print(f"correctly flagged with at least one pathology: {abnormal_detected}")
print(f"abnormal recall rate: {abnormal_recall_rate:.2%}")
print(f"cakupan validasi: {covered}/{len(df_dev)} dev samples ({uncovered} tidak masuk kategori manapun — kemungkinan semua label uncertain/NaN tanpa No Finding=1)")

In [ ]:
print("\ninterpretation:")
if normal_detection_rate < 0.90:
    print("  -> warning: threshold too low. many normal cases are being over-called as abnormal.")
elif normal_detection_rate > 0.99 and abnormal_recall_rate < 0.50:
    print("  -> warning: threshold too high. many abnormal cases are being missed.")
else:
    print("  -> threshold balanced well between preserving normal cases and detecting abnormalities.")

In [ ]:
# test set evaluation with global vs per-class thresholds for each checkpoint
def evaluate_test_with_thresholds(test_preds, test_labels, tuning, label_cols, other_indices, label):
    """evaluate test set using global threshold and per-class thresholds."""
    test_labels_other = test_labels[:, other_indices]
    preds_global = (test_preds[:, other_indices] >= tuning["best_th"]).astype(int)
    preds_per_class = np.zeros_like(test_labels_other)
    for local_i, global_i in enumerate(other_indices):
        th = tuning["per_class_th"][label_cols[global_i]]
        preds_per_class[:, local_i] = (test_preds[:, global_i] >= th).astype(int)

    bal_acc_global = balanced_accuracy_score(test_labels_other, preds_global)
    bal_acc_per_class = balanced_accuracy_score(test_labels_other, preds_per_class)

    print(f"[{label}] test bal_acc - global threshold ({tuning['best_th']:.2f}): {bal_acc_global:.4f}")
    print(f"[{label}] test bal_acc - per-class threshold: {bal_acc_per_class:.4f}")
    return {"bal_acc_global": bal_acc_global, "bal_acc_per_class": bal_acc_per_class, "preds_global": preds_global, "preds_per_class": preds_per_class}

In [ ]:
test_eval_f1 = evaluate_test_with_thresholds(
    results_best_f1["test_preds"], 
    results_best_f1["test_labels"], 
    tuning_f1, 
    label_cols, 
    other_indices, 
    "best-by-f1"
)
test_eval_prauc = evaluate_test_with_thresholds(
    results_best_prauc["test_preds"], 
    results_best_prauc["test_labels"], 
    tuning_prauc, 
    label_cols, 
    other_indices, 
    "best-by-prauc"
)

# final classification report using best-by-f1 + per-class threshold
print(f"\ntest set classification report (best-by-f1, per-class threshold)")
print(classification_report(
    results_best_f1["test_labels"][:, other_indices], test_eval_f1["preds_per_class"],
    target_names=[label_cols[i] for i in other_indices], zero_division=0
))

# final classification report using best-by-prauc + per-class threshold
print(f"\ntest set classification report (best-by-prauc, per-class threshold)")
print(classification_report(
    results_best_prauc["test_labels"][:, other_indices], test_eval_prauc["preds_per_class"],
    target_names=[label_cols[i] for i in other_indices], zero_division=0
))

In [ ]:
# prompt injection function
def build_chexpert_findings_prompt(row_probs, per_class_th, label_cols, other_indices):
    """
    format the 'chexpert findings: ...' string using per-pathology thresholds.
    a pathology is included iff its probability >= its own threshold.
    """
    active = [label_cols[i].replace('_label', '') for i in other_indices if row_probs[i] >= per_class_th[label_cols[i]]]
    if len(active) == 0:
        return "chexpert findings: no significant findings"
    return f"chexpert findings: {', '.join(active)}"

# best-by-prauc
print("\ncontoh prompt injection (per-class threshold, best-by-prauc):")
for i in range(3):
    print(f"  sample {i}: {build_chexpert_findings_prompt(dev_probs_prauc[i], tuning_prauc['per_class_th'], label_cols, other_indices)}")

# best-by-f1
print("\ncontoh prompt injection (per-class threshold, best-by-f1, comparison only):")
for i in range(3):
    print(f"  sample {i}: {build_chexpert_findings_prompt(dev_probs_f1[i], tuning_f1['per_class_th'], label_cols, other_indices)}")

In [ ]:
# example usage on dev set (best-by-prauc)
print("\nexamples of prompt injection (per-class threshold, best-by-prauc):")
for i in range(3):
    print(f"  sample {i}: {build_chexpert_findings_prompt(dev_probs_prauc[i], tuning_prauc['per_class_th'], label_cols, other_indices)}")

# comparison using best-by-f1 thresholds (not used in production, kept for reference)
print("\nexamples of prompt injection (per-class threshold, best-by-f1):")
for i in range(3):
    print(f"  sample {i}: {build_chexpert_findings_prompt(dev_probs_f1[i], tuning_f1['per_class_th'], label_cols, other_indices)}")

## 7. Summary of Stage-B

In [ ]:
summary = f"""
====================================================================
stage-b training summary
====================================================================
experiment: {config['experiment']}
stage: {config['stage']}
device: {device}

dataset:
  - train: {len(df_train)} samples
  - dev: {len(df_dev)} samples
  - test: {len(df_test)} samples

model:
  - backbone: biovil-t
  - vision dimension: {config['vision_dim']}
  - pooling: {config['model']['pooling']}
  - hidden dim: {config['model']['hidden_dim']}
  - dropout: {config['model']['dropout']}
  - total params: {total_params/1e6:.2f}m
  - trainable params (final epoch): {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.2f}m

training:
  - loss: {config['loss']['type']}+pos_weight
  - batch size: {config['train']['batch_size']} (effective: {config['train']['batch_size'] * config['train']['grad_accum_steps']})
  - learning rate (head / backbone): {config['train']['lr']} / {config['train']['lr'] * config['train']['backbone_lr_mult']}
  - weight decay: {config['train']['weight_decay']}
  - unfreeze schedule: {config['train']['unfreeze_schedule']}
  - use_amp: {config['train']['use_amp']}
  - epochs run: {len(train_losses)}
  - best val macro-f1 (threshold=0.5): {best_val_macro_f1:.4f}
  - best val macro pr-auc: {best_val_macro_prauc:.4f}

threshold tuning (on dev set, excluding no finding, tuned separately per checkpoint):
  - best-by-f1    : global threshold={tuning_f1['best_th']:.2f} (macro f1={tuning_f1['best_macro_f1']:.4f}) | per-class macro f1={tuning_f1['macro_f1_per_class_th']:.4f}
  - best-by-prauc : global threshold={tuning_prauc['best_th']:.2f} (macro f1={tuning_prauc['best_macro_f1']:.4f}) | per-class macro f1={tuning_prauc['macro_f1_per_class_th']:.4f}

normal detection rate (dev set, genuinely normal cases, best-by-prauc + per-class threshold):
  - correctly flagged as normal: {correctly_flagged_normal}/{total_normal}
  - normal detection rate: {normal_detection_rate:.2%}
  - abnormal recall rate: {abnormal_recall_rate:.2%}

test performance:
  - best-by-f1 checkpoint   : bal_acc={results_best_f1['bal_acc']:.4f}, macro_prauc={results_best_f1['macro_prauc']:.4f}, bal_acc 95% CI=[{results_best_f1['bal_acc_ci'][0]:.4f}, {results_best_f1['bal_acc_ci'][1]:.4f}]
  - best-by-prauc checkpoint: bal_acc={results_best_prauc['bal_acc']:.4f}, macro_prauc={results_best_prauc['macro_prauc']:.4f}, bal_acc 95% CI=[{results_best_prauc['bal_acc_ci'][0]:.4f}, {results_best_prauc['bal_acc_ci'][1]:.4f}]
  - best-by-f1    + global threshold   : bal_acc={test_eval_f1['bal_acc_global']:.4f}
  - best-by-f1    + per-class threshold: bal_acc={test_eval_f1['bal_acc_per_class']:.4f}
  - best-by-prauc + global threshold   : bal_acc={test_eval_prauc['bal_acc_global']:.4f}
  - best-by-prauc + per-class threshold: bal_acc={test_eval_prauc['bal_acc_per_class']:.4f}

downstream candidate (default assumption, see "open decision" in section 0):
  best-by-prauc checkpoint + its per-class thresholds drive prompt injection.
  best-by-f1 checkpoint + its per-class thresholds are kept as a comparison artifact only.
output directory: {config['output']['checkpoint_dir']}
====================================================================
"""

print(summary)

with open(os.path.join(config['output']['log_dir'], 'training_summary.txt'), 'w') as f:
    f.write(summary)

print("\nall done. stage-b training and evaluation complete (exp_002).")